# 작업 6.1 Estimator 옵션 설정하기

**개요:** 이 노트북에서는 실행 동작과 오류 완화 기법을 제어하는 다양한 Estimator 옵션을 다룹니다.

In [1]:
# 설정: 필요한 라이브러리 가져오기
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator, EstimatorOptions as Options
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime.options import MeasureNoiseLearningOptions,LayerNoiseLearningOptions,ZneOptions

print("라이브러리를 성공적으로 가져왔습니다.")

라이브러리를 성공적으로 가져왔습니다.


## 목표 1: Estimator 옵션

`EstimatorV2`의 `Options` 클래스는 `Estimator` 실행에 대한 다양한 설정값을 제공합니다. 이 설정은 성능, 정확도, 자원 사용량에 영향을 줍니다.

### 속성


| 속성 | 설명 | 비고 |
|-----------|-------------|----------------|
| **`_VERSION`** | 내부 버전 추적 |  |
| **`max_execution_time`** | 작업 실행의 최대 시간(초) |  |
| **`environment`** | 실행 환경 메타데이터 | `EnvironmentOptions` 객체 |
| **`simulator`** | 시뮬레이터 전용 옵션 | `{'noise_model': None, 'seed_simulator': None}` |
| **`default_precision`** | 기대값의 목표 정밀도 | 기본값은 `0.015625` (1/√4096)인 부동소수점 값 |
| **`default_shots`** | 회로의 기본 실행 횟수 | 정수, 기본값 없음 |
| **`resilience_level`** | 오류 완화 수준 | `0`(없음)부터 `2`(중간)까지 |
| **`seed_estimator`** | 재현 가능한 결과를 위한 난수 시드 | 정수 |
| **`dynamical_decoupling`** | 동적 디커플링 활성화/설정 | `DynamicalDecouplingOptions` 객체 |
| **`resilience`** | 고급 resilience 설정 | `Options.ResilienceOptionsV2` 객체 |
| **`execution`** | 실행 관련 설정 | `ExecutionOptionsV2 ` 객체 |
| **`twirling`** | 게이트 트월링 설정 | `Options.TwirlingOptions` 객체 |
| **`experimental`** | 실험적 기능 | 실험 설정을 담은 딕셔너리 |

In [2]:
# 예시: Estimator 옵션 설정하기

# 서비스 초기화 (자격 증명은 이미 저장되어 있다고 가정)
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# 간단한 테스트 회로 생성
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
observable = SparsePauliOp("ZZ")

# 사용자 정의 설정으로 Options 객체 생성
options = Options()

# 기본 실행 파라미터 설정
options.max_execution_time = 300
options.default_shots = 2048 
options.default_precision = 0.01
options.seed_estimator = 12345

# resilience 수준 설정 (0=없음, 1=기본, 2=중간, 3=최대)
options.resilience_level = 2  # 중간 수준 오류 완화

print(f"최대 실행 시간: {options.max_execution_time}초")
print(f"기본 shots 수: {options.default_shots}")
print(f"기본 정밀도: {options.default_precision}")
print(f"Resilience 수준: {options.resilience_level}\n")

# 사용자 정의 옵션으로 Estimator 생성
estimator = Estimator(mode=backend, options=options)

print("옵션 업데이트\n")

# 초기 옵션 생성
initial_options = Options()
initial_options.default_shots = 1024
initial_options.max_execution_time = 300

print(f"초기 shots 수: {initial_options.default_shots}")
print(f"초기 최대 시간: {initial_options.max_execution_time}")

# 새 값으로 옵션 갱신
initial_options.update(default_shots=4096, max_execution_time=900)

print(f"업데이트된 shots 수: {initial_options.default_shots}")
print(f"업데이트된 최대 시간: {initial_options.max_execution_time}\n")

qiskit_runtime_service.__init__:WARNING:2026-04-14 19:44:00,748: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (internal, premium), the available account instances are: Solutions Demo internal fleet, Educational premium fleet, Educational internal fleet, Solutions Notebooks internal fleet, Solutions Notebooks premium fleet, Solutions Demo premium fleet, Solutions Demo internal fleet, Solutions Notebooks premium fleet, Educational premium fleet, Solutions Notebooks internal fleet, Solutions Demo premium fleet, Educational internal fleet. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-04-14 19:44:08,369: Loading instance: Solutions Demo internal fleet, plan: internal
qiskit_runtime_servi

최대 실행 시간: 300초
기본 shots 수: 2048
기본 정밀도: 0.01
Resilience 수준: 2

옵션 업데이트

초기 shots 수: 1024
초기 최대 시간: 300
업데이트된 shots 수: 4096
업데이트된 최대 시간: 900



### 메서드

#### update

여러 옵션을 한 번에 갱신합니다. `Estimator` 인스턴스를 만든 뒤 옵션을 수정하거나, 설정 변경을 적용할 때 유용합니다.

In [3]:
options = Options()
options.update(
    default_shots=4096,
    max_execution_time=1200,
    resilience_level=2
)

## 목표 2: Twirling 옵션

트월링(twirling, 또는 randomized compiling)은 서로 동등하지만 다른 게이트 시퀀스들을 평균내어, 코히런트 오류를 완화하기 쉬운 확률적 오류로 바꾸는 기법입니다.

| 속성 | 설명 | 비고 |
|-----------|-------------|----------------|
| **`enable_gates`** | 2큐비트 Clifford 게이트 트월링 사용 여부 | bool, 기본값 `False` |
| **`enable_measure`** | 측정 명령에 대한 트월링 사용 여부 | Estimator에서는 `True`, Sampler에서는 `False` |
| **`num_randomizations`** | 트월링 시 사용할 무작위 샘플 수 | 정수 또는 `'auto'` |
| **`shots_per_randomization`** | 각 무작위 샘플마다 실행할 shot 수 | 정수 또는 `'auto'` |
| **`strategy`** | 큐비트 트월링 전략 | `active`, `active-circuit`, `active-accum`, `all`, 기본값 `active-accum` |

## 목표 3: Resilience 옵션

`Resilience` 옵션은 `EstimatorV2`에서의 오류 완화 기법을 제어합니다. 이러한 설정은 잡음이 있는 하드웨어에서 양자 계산의 정확도를 높이는 데 도움을 줍니다.

### 속성


| 속성 | 설명 | 비고 |
|-----------|-------------|---------|
| **`measure_mitigation`** | 측정 오류를 보정 | 기본값 `True` |
| **`measure_noise_learning`** | 측정 잡음 모델을 학습 | `MeasureNoiseLearningOptions` 객체 |
| **`zne_mitigation`** | Zero-Noise Extrapolation 사용 | 기본값 `False` |
| **`zne`** | ZNE 전용 설정 | `ZneOptions` 객체 |
| **`pec_mitigation`** | Probabilistic Error Cancellation 사용 | 기본값 `False` |
| **`pec`** | PEC 전용 설정 | `PecOptions` |
| **`layer_noise_learning`** | 레이어별 잡음을 학습 | `LayerNoiseLearningOptions` 객체 |
| **`layer_noise_model`** | 사용자 정의 레이어 잡음 모델 | `NoiseLearnerResult` 또는 `Sequence[LayerError]` |

In [4]:
# 예시: Resilience 옵션 설정하기

# 방법 1: resilience_level 사용
levels = [0, 1, 2]
descriptions = [
    "오류 완화 없음",
    "최소 수준의 측정 오류 완화",
    "중간 수준 완화"
]

for level, desc in zip(levels, descriptions):
    options = Options()
    options.resilience_level = level
    print(f"Level {level}: {desc}")

# 방법 2: Resilience 옵션으로 직접 제어

options = Options()

# 개별 resilience 기법 설정
# 기본 readout 보정
options.resilience.measure_mitigation = True
# 측정 잡음 학습
options.resilience.measure_noise_learning = MeasureNoiseLearningOptions()
# Zero-Noise Extrapolation 활성화
options.resilience.zne_mitigation = True
# PEC 비활성화 (계산 비용이 큼)
options.resilience.pec_mitigation = False
# 레이어 의존 잡음 학습
options.resilience.layer_noise_learning = LayerNoiseLearningOptions() 

# ZNE 세부 설정
options.resilience.zne = ZneOptions(noise_factors=[1.0, 2.0, 3.0], extrapolator="exponential")

print("설정된 resilience 값:")
print(f"  측정 완화: {options.resilience.measure_mitigation}")
print(f"  ZNE 완화: {options.resilience.zne_mitigation}")
print(f"  PEC 완화: {options.resilience.pec_mitigation}")
print(f"  ZNE noise factors: {options.resilience.zne.noise_factors}")
print(f"  ZNE 외삽기: {options.resilience.zne.extrapolator}\n")


# 단계 1: 최소 resilience
quick_test = Options()
quick_test.resilience_level = 0
quick_test.default_shots = 256
print(f"  Resilience: {quick_test.resilience_level} (없음)")
print(f"  Shots: {quick_test.default_shots}")

# 단계 2: 중간 resilience
moderate = Options()
moderate.resilience_level = 1
moderate.default_shots = 1024
print(f"  Resilience: {moderate.resilience_level} (기본)")
print(f"  Shots: {moderate.default_shots}")

# 단계 3: 높은 resilience
high = Options()
high.resilience_level = 2
high.default_shots = 4096
print(f"  Resilience: {high.resilience_level} (고급)")
print(f"  Shots: {high.default_shots}")

Level 0: 오류 완화 없음
Level 1: 최소 수준의 측정 오류 완화
Level 2: 중간 수준 완화
설정된 resilience 값:
  측정 완화: True
  ZNE 완화: True
  PEC 완화: False
  ZNE noise factors: [1.0, 2.0, 3.0]
  ZNE 외삽기: exponential

  Resilience: 0 (없음)
  Shots: 256
  Resilience: 1 (기본)
  Shots: 1024
  Resilience: 2 (고급)
  Shots: 4096


## 목표 4: ZNE 옵션

Zero-Noise Extrapolation(ZNE)은 다음과 같은 오류 완화 기법입니다.
1. 증폭된 잡음 수준에서 회로를 실행합니다.
2. 서로 다른 잡음 배수(noise factor)에서 결과를 측정합니다.
3. 결과를 무잡음 한계(zero-noise limit)로 외삽합니다.

### 속성


| 속성 | 설명 | 옵션 |
|-----------|-------------|---------|
| **`amplifier`** | 잡음 증폭 방식 | `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea` |
| **`noise_factors`** | 잡음 증폭 수준 | 실수 리스트 (예: `[1.0, 2.0, 3.0]`) |
| **`extrapolator`** | 외삽 방식 | `linear`, `exponential`, `double_exponential`, `polynomial_degree_1`, `polynomial_degree_2`, `polynomial_degree_3`, `polynomial_degree_4`, `polynomial_degree_5`, `polynomial_degree_6`, `polynomial_degree_7`, `fallback` |
| **`extrapolated_noise_factors`** | 피팅에 사용됨 | `noise_factors`로부터 유도 |

In [5]:
# 예시: ZNE 옵션 설정하기

print("=== Zero-Noise Extrapolation (ZNE) 설정 ===\n")

# 먼저 resilience 설정에서 ZNE 활성화
options = Options()
options.resilience.zne_mitigation = True

# ZNE 파라미터 설정
print("ZNE 설정 예시:\n")

# gate folding을 사용하는 기본 ZNE
options.resilience.zne.amplifier = "gate_folding"
options.resilience.zne.noise_factors = [1.0, 2.0, 3.0]
options.resilience.zne.extrapolator = "linear"

print(f"사례 1:")
print(f"  증폭기: {options.resilience.zne.amplifier}")
print(f"  Noise factors: {options.resilience.zne.noise_factors}")
print(f"  외삽기: {options.resilience.zne.extrapolator}")

# 더 높은 정확도를 위한 고급 ZNE
options.resilience.zne.amplifier = "gate_folding"
options.resilience.zne.noise_factors = [1.0, 1.5, 2.0, 3.0, 4.0]
options.resilience.zne.extrapolator = "exponential"

print(f"사례 2:")
print(f"  증폭기: {options.resilience.zne.amplifier}")
print(f"  Noise factors: {options.resilience.zne.noise_factors}")
print(f"  외삽기: {options.resilience.zne.extrapolator}")

# ZNE 설정 생성
zne_config = Options()

# ZNE 활성화 및 설정
zne_config.resilience.zne_mitigation = True
zne_config.resilience.zne.amplifier = "gate_folding"
zne_config.resilience.zne.noise_factors = [1.0, 2.0, 3.0, 4.0]
zne_config.resilience.zne.extrapolator = "polynomial_degree_1"  # 다항식 외삽


# ZNE용 shots 수와 실행 시간 설정
zne_config.default_shots = 2048
zne_config.max_execution_time = 600  # 10분

print("전체 ZNE 설정:")
print(f"  ZNE 활성화: {zne_config.resilience.zne_mitigation}")
print(f"  증폭기: {zne_config.resilience.zne.amplifier}")
print(f"  Noise factors: {zne_config.resilience.zne.noise_factors}")
print(f"  외삽기: {zne_config.resilience.zne.extrapolator}")
print(f"  factor당 shots 수: {zne_config.default_shots}")
print(f"  총 예상 shots 수: {zne_config.default_shots * len(zne_config.resilience.zne.noise_factors)}")
print(f"  최대 실행 시간: {zne_config.max_execution_time}초")

=== Zero-Noise Extrapolation (ZNE) 설정 ===

ZNE 설정 예시:

사례 1:
  증폭기: gate_folding
  Noise factors: [1.0, 2.0, 3.0]
  외삽기: linear
사례 2:
  증폭기: gate_folding
  Noise factors: [1.0, 1.5, 2.0, 3.0, 4.0]
  외삽기: exponential
전체 ZNE 설정:
  ZNE 활성화: True
  증폭기: gate_folding
  Noise factors: [1.0, 2.0, 3.0, 4.0]
  외삽기: polynomial_degree_1
  factor당 shots 수: 2048
  총 예상 shots 수: 8192
  최대 실행 시간: 600초


---
## 요약
---

이 노트북에서는 다음을 다뤘습니다.

## Estimator 옵션 설정:

1. **Estimator 옵션**은 `default_shots`, `default_precision`, `resilience_level` 같은 핵심 설정을 포함합니다.
2. **Twirling**은 코히런트 오류를 확률적 오류로 바꿉니다.
3. **Resilience 활성화**는 `resilience_level`에 따라 서로 다른 오류 완화 기법을 적용합니다.
4. **Zero-noise extrapolation**은 증폭된 잡음 수준에서 회로를 실행한 뒤 무잡음 한계로 외삽합니다.

---

## 연습 문제

**1) `EstimatorOptions`에서 `resilience_level`을 설정하는 주된 목적은 무엇인가요?**

A) 백엔드에서의 회로 실행을 제어하기 위해.

B) 트랜스파일 단계의 resilience 최적화 수준을 선택하기 위해.

C) 오류 완화 기법을 활성화하고 조정하기 위해.

D) 배치 처리와 세션 동작을 설정하기 위해.

**정답:**
<details> <br/>

C) `resilience_level`은 어떤 오류 완화 전략(트월링, ZNE, ...)을 활성화할지, 그리고 그것을 어떻게 적용할지를 결정합니다.

</details>

---

**2) `active`와 `active-circuit` 트월링 전략의 핵심 차이는 무엇인가요?**

A) `active`는 측정을 트월링하고, `active-circuit`는 측정을 트월링하지 않는다.

B) `active`는 각 레이어의 명령에 대해 개별적으로 트월링을 적용하고, `active-circuit`는 각 트월링 레이어에서 회로 전체에 걸친 명령 큐비트들의 합집합에 대해 트월링을 적용한다.

C) `active`는 레이어 간 트월링을 누적하고, `active-circuit`는 각 레이어 뒤에 초기화한다.

D) `active`는 시뮬레이터에서만 지원된다.

**정답:**
<details> <br/>

B) `'active'`를 사용할 때는 각 트월링 레이어의 개별 명령 큐비트만 트월링되지만, `'active-circuit'`를 사용할 때는 회로 전체에서 등장하는 명령 큐비트들의 합집합이 각 트월링 레이어에서 트월링됩니다.

</details>

---

**3) 5개의 noise factor와 다항식 외삽을 사용하도록 ZNE를 설정하려면 아래 코드를 완성하세요:**

```python
from qiskit_ibm_runtime import EstimatorOptions as Options

options = Options()

# ZNE 완화 활성화
options.resilience.______ = True

# ZNE 파라미터 설정
options.resilience.zne.amplifier = "gate_folding"
options.resilience.zne.______ = [1.0, 1.5, 2.0, 3.0, 4.0]
options.resilience.zne.extrapolator = "______"
```

어느 선택지가 세 개의 빈칸을 모두 올바르게 채우나요?

A) zne_enable, noise_levels, polynomial

B) zne, factors, polynomial_degree_2

C) zne_mitigation, noise_factors, polynomial_degree_2

D) enable_zne, noise_factors, poly_2

**정답:**
<details> <br/>

C) 올바른 속성 이름은 `zne_mitigation`, `noise_factors`, `polynomial_degree_2` 입니다.
</details>

----